## 415_check_WD_object
* [#415](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/415)
* denna Notebook [415_check_WD_object](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/415_check_WD_object.ipynb)

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-18 10:51


In [2]:
import requests

url = "https://map.stockholmarchipelagotrail.com/data/geojson/poi-concordance.json"
data = requests.get(url).json()

wikidata_ids = [
    k.replace("wikidata:", "")
    for k in data["satIdOf"].keys()
    if k.startswith("wikidata:")
]

print(f"{len(wikidata_ids)} Wikidata-objekt")
print(wikidata_ids[:10])

199 Wikidata-objekt
['Q106653509', 'Q106653538', 'Q106680381', 'Q107290890', 'Q121361560', 'Q121433209', 'Q121433242', 'Q121715058', 'Q121754355', 'Q130338597']


In [3]:
# Issue #415
# SAT Wikidata QA checker
#
# Checks all Wikidata references in poi-concordance.json
#
# Reports:
#   - Redirected Wikidata objects (merged)
#   - Deleted Wikidata objects
#   - Related SAT object
#   - Related OSM objects
#
# Output:
#   output/issue_415_wikidata_quality_report_YYYYMMDD.json
#   output/issue_415_wikidata_issues_YYYYMMDD.csv
#   output/issue_415_wikidata_issues_YYYYMMDD.html
#
# Notebook:
#   Displays clickable table with:
#     Wikidata
#     Redirect target
#     SAT API link
#     OSM links

import json
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from IPython.display import HTML, display

ISSUE = 415

CONCORDANCE_URL = (
    "https://map.stockholmarchipelagotrail.com/data/geojson/poi-concordance.json"
)

# --------------------------------------------------
# Setup
# --------------------------------------------------

today = datetime.now().strftime("%Y%m%d")

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

json_file = (
    output_dir /
    f"issue_{ISSUE}_wikidata_quality_report_{today}.json"
)

csv_file = (
    output_dir /
    f"issue_{ISSUE}_wikidata_issues_{today}.csv"
)

html_file = (
    output_dir /
    f"issue_{ISSUE}_wikidata_issues_{today}.html"
)

print("Loading concordance...")

concordance = requests.get(CONCORDANCE_URL).json()

sat_of = concordance["satIdOf"]

# --------------------------------------------------
# Build indexes
# --------------------------------------------------

wikidata_to_sat = {}
sat_to_osm = {}

for key, sat_id in sat_of.items():

    if key.startswith("wikidata:"):
        qid = key.replace("wikidata:", "")
        wikidata_to_sat[qid] = sat_id

    elif key.startswith("osm:"):
        sat_to_osm.setdefault(
            sat_id,
            []
        ).append(key)

print(
    f"Found {len(wikidata_to_sat)} Wikidata objects"
)

# --------------------------------------------------
# Check Wikidata
# --------------------------------------------------

alive = []
redirects = []
deleted = []
errors = []

for idx, qid in enumerate(
    sorted(wikidata_to_sat.keys()),
    start=1
):

    print(
        f"{idx}/{len(wikidata_to_sat)} {qid}",
        end="\r"
    )

    sat_id = wikidata_to_sat[qid]

    try:

        url = (
            f"https://www.wikidata.org/wiki/"
            f"Special:EntityData/{qid}.json"
        )

        r = requests.get(
            url,
            timeout=30,
            headers={
                "User-Agent":
                "Stockholm Archipelago Trail QA"
            }
        )

        osm_objects = sat_to_osm.get(
            sat_id,
            []
        )

        if r.status_code == 404:

            deleted.append({
                "status": "deleted",
                "qid": qid,
                "target": "",
                "sat_id": sat_id,
                "osm": osm_objects
            })

            continue

        if r.status_code != 200:

            errors.append({
                "qid": qid,
                "status_code": r.status_code
            })

            continue

        entities = r.json()["entities"]

        returned_qid = next(
            iter(entities.keys())
        )

        if returned_qid != qid:

            redirects.append({
                "status": "redirect",
                "qid": qid,
                "target": returned_qid,
                "sat_id": sat_id,
                "osm": osm_objects
            })

        else:
            alive.append(qid)

    except Exception as e:

        errors.append({
            "qid": qid,
            "error": str(e)
        })

    time.sleep(0.1)

print()
print("Done")

# --------------------------------------------------
# Create dataframe
# --------------------------------------------------

rows = []

for item in redirects + deleted:

    osm_urls = []

    for osm in item["osm"]:

        _, osm_type, osm_id = osm.split(":")

        osm_urls.append(
            f"https://www.openstreetmap.org/"
            f"{osm_type}/{osm_id}"
        )

    rows.append({
        "Status": item["status"],
        "Wikidata": item["qid"],
        "Target": item["target"],
        "SAT": item["sat_id"],
        "OSM Count": len(osm_urls),
        "OSM Links": "\n".join(osm_urls)
    })

df = pd.DataFrame(rows)

# --------------------------------------------------
# Save CSV
# --------------------------------------------------

df.to_csv(
    csv_file,
    index=False
)

# --------------------------------------------------
# Save JSON
# --------------------------------------------------

report = {
    "alive": len(alive),
    "redirects": redirects,
    "deleted": deleted,
    "errors": errors,
}

with open(
    json_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

# --------------------------------------------------
# HTML helpers
# --------------------------------------------------

def wd_link(qid):

    if not qid:
        return ""

    return (
        f'<a href="https://www.wikidata.org/wiki/{qid}" '
        f'target="_blank">{qid}</a>'
    )

def sat_link(sat_id):

    if not sat_id:
        return ""

    url = (
        "https://map.stockholmarchipelagotrail.com/api/objects/"
        f"{sat_id}"
    )

    return (
        f'<a href="{url}" target="_blank">'
        f'{sat_id}'
        f'</a>'
    )

def osm_links(text):

    if not text:
        return ""

    html = []

    for url in text.split("\n"):

        parts = url.split("/")

        html.append(
            f'<a href="{url}" target="_blank">'
            f'{parts[-2]}/{parts[-1]}'
            f'</a>'
        )

    return "<br>".join(html)

# --------------------------------------------------
# Notebook table
# --------------------------------------------------

if len(df):

    html_df = df.copy()

    html_df["Wikidata"] = (
        html_df["Wikidata"]
        .apply(wd_link)
    )

    html_df["Target"] = (
        html_df["Target"]
        .apply(wd_link)
    )

    html_df["SAT"] = (
        html_df["SAT"]
        .apply(sat_link)
    )

    html_df["OSM Links"] = (
        html_df["OSM Links"]
        .apply(osm_links)
    )

    html = html_df.to_html(
        escape=False,
        index=False
    )

    with open(
        html_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(html)

    display(
        HTML(html)
    )

# --------------------------------------------------
# Summary
# --------------------------------------------------

print()
print("Summary")
print("--------------------------------")
print("Alive     :", len(alive))
print("Redirects :", len(redirects))
print("Deleted   :", len(deleted))
print("Errors    :", len(errors))

print()
print("Files")
print("--------------------------------")
print(json_file)
print(csv_file)
print(html_file)


Loading concordance...
Found 199 Wikidata objects
199/199 Q636155978
Done


Status,Wikidata,Target,SAT,OSM Count,OSM Links
redirect,Q134871089,Q106678630,sat:poi:978hx,1,node/12905547872
redirect,Q135040652,Q134692920,sat:poi:zqc6r,1,node/12881274960
redirect,Q136337450,Q135110110,sat:poi:pvz9y,1,node/13157558827
deleted,Q134605030,,sat:poi:c8je6,1,node/3431736753
deleted,Q134607564,,sat:poi:b7nxb,1,node/1392309442
deleted,Q134607783,,sat:poi:mxxwb,1,node/863394662
deleted,Q134717347,,sat:poi:g3k29,1,node/12090393146
deleted,Q134717362,,sat:poi:kkevg,1,node/9936118752
deleted,Q134726995,,sat:poi:b3bvp,1,node/4958553279
deleted,Q134985579,,sat:poi:nsjj5,1,node/5728811330



Summary
--------------------------------
Alive     : 173
Redirects : 3
Deleted   : 18
Errors    : 5

Files
--------------------------------
output/issue_415_wikidata_quality_report_20260618.json
output/issue_415_wikidata_issues_20260618.csv
output/issue_415_wikidata_issues_20260618.html


In [4]:
end_time = time.time()
duration = end_time - start_time

print(f"Fini§shed in {duration:.2f} seconds.")


Fini§shed in 98.23 seconds.
